In [73]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler

In [74]:
continuous_features = ['latDev', 'longDev', 'elevSat', 'loraTP', 'loraSF', 'doppler', 'alt', 'raan', 'srcId']
cols_to_drop        = ['dstId', 'srcSat', 'dstSat', 'loraCF', 'loraBW', 'loraCR', 'satId'] #, 'srcId']

-------> Processing Seed 1/10 ---
-------> Processing Seed 2/10 ---
-------> Processing Seed 3/10 ---
-------> Processing Seed 4/10 ---
-------> Processing Seed 5/10 ---
-------> Processing Seed 6/10 ---
-------> Processing Seed 7/10 ---
-------> Processing Seed 8/10 ---
-------> Processing Seed 9/10 ---
-------> Processing Seed 10/10 ---


In [77]:
def create_sequences_data(df, scaler, seq_length):    
    df = df.drop(columns=cols_to_drop)

    # Sort by id_simulaiton and time
    df = df.sort_values(by=['id_simulation', 'srcId','time']).reset_index(drop=True)
    
    # Apply StandardScaler         
    #df[continuous_features] = scaler.transform(df[continuous_features])

    # Process data
    seq_X   = []
    label_y = []
    for sim_id, group_df in df.groupby(by=['id_simulation', 'srcId']):        
        num_features_array  = group_df[continuous_features].values
        time_array          = group_df['time'].values
        target_array        = group_df['rcvOk'].values
        num_packets         = len(group_df)
        
        # Discard if the simulation has less than SEQ_LENGHTS
        if num_packets < seq_length:
            continue
                    
        # Create sliding windows - Ventanas deslizantes
        for i in range(num_packets - seq_length + 1):
            # Select the window        
            window_num      = num_features_array[i : i + seq_length]
            ###window_cat      = cat_features_array[i : i + SEQ_LENGTH].reshape(-1, 1)
            window_times    = time_array[i : i + seq_length]
            window_targets  = target_array[i : i + seq_length]
            
            # El paquete central a predecir es el último de la ventana
            # The prediction should be the last elemento of the window
            target_idx      = seq_length - 1
            label_target    = window_targets[target_idx]
            time_target     = window_times[target_idx]
                        
            # Si un paquete ocurrió en el mismo segundo que el objetivo, delta_t = 0        
            #delta_t = (window_times - time_target).reshape(-1, 1)
            delta_t = (time_target - window_times).reshape(-1, 1)
            #delta_t =  time_target - window_times
            #print('delta_t: ', delta_t)
            
            #window_X = np.hstack((window_num, window_cat, delta_t))
            window_X = np.hstack((window_num, delta_t))

            seq_X.append(window_X)
            label_y.append(label_target)
    #print("seq_X: ", len(seq_X), "label_y: ", len(label_y))
    #print(seq_X)
    return seq_X, label_y
    #X = torch.tensor(np.array(seq_X), dtype=torch.float32)
    #y = torch.tensor(np.array(label_y), dtype=torch.float32)

In [118]:
(77.62/22.38) ** 0.5

1.8623305951831266

In [78]:
base_path  = 'data/repetitions/'
num_seeds  = 10
for seed in range(num_seeds):
    print(f"-------> Processing Seed {seed+1}/{num_seeds} ---")

    # Construcción dinámica de nombres de archivos
    path_train = os.path.join(base_path, f'seed_{seed}_final_train_data_transmission.csv')
    path_test  = os.path.join(base_path, f'seed_{seed}_final_test_data_transmission.csv')
    path_val   = os.path.join(base_path, f'seed_{seed}_final_val_data_transmission.csv')

    df_train = pd.read_csv(path_train)
    df_test  = pd.read_csv(path_test)
    df_val   = pd.read_csv(path_val)

    X_train_features = df_train[continuous_features].values
    X_test_features = df_test[continuous_features].values
    X_val_features = df_val[continuous_features].values


    scaler = StandardScaler()
    scaler.fit(X_train_features)
    
    seq_length = 8
    #X_train, y_train = 
    seq_X, label_y = create_sequences_data(df_train, scaler, seq_length)
    #X_test , y_test  = create_sequences_data(df_test, scaler, seq_length)
    #X_val  , y_val   = create_sequences_data(df_val, scaler, seq_length)    

-------> Processing Seed 1/10 ---
-------> Processing Seed 2/10 ---
-------> Processing Seed 3/10 ---
-------> Processing Seed 4/10 ---
-------> Processing Seed 5/10 ---
-------> Processing Seed 6/10 ---
-------> Processing Seed 7/10 ---
-------> Processing Seed 8/10 ---
-------> Processing Seed 9/10 ---
-------> Processing Seed 10/10 ---


In [79]:
print(len(seq_X))
seq_X[4]

43680


array([[ 5.280190e+01, -1.403250e-01,  3.420890e+00,  1.995300e-02,
         1.200000e+01,  1.822140e+04,  6.000000e+02,  1.700000e+02,
         0.000000e+00,  1.147922e+02],
       [ 5.280190e+01, -1.403250e-01,  4.377740e+00,  1.995300e-02,
         1.200000e+01,  1.808060e+04,  6.000000e+02,  1.700000e+02,
         0.000000e+00,  1.003004e+02],
       [ 5.280190e+01, -1.403250e-01,  5.540480e+00,  1.995300e-02,
         1.200000e+01,  1.785030e+04,  6.000000e+02,  1.700000e+02,
         0.000000e+00,  8.331200e+01],
       [ 5.280190e+01, -1.403250e-01,  7.244680e+00,  1.995300e-02,
         1.200000e+01,  1.738470e+04,  6.000000e+02,  1.700000e+02,
         0.000000e+00,  5.958700e+01],
       [ 5.280190e+01, -1.403250e-01,  8.264760e+00,  1.995300e-02,
         1.200000e+01,  1.714830e+04,  6.000000e+02,  1.700000e+02,
         0.000000e+00,  4.601500e+01],
       [ 5.280190e+01, -1.403250e-01,  9.126840e+00,  1.995300e-02,
         1.200000e+01,  1.677860e+04,  6.000000e+02,  1.7

In [114]:
print(len(seq_X))
#
print(np.array2string(seq_X[52], suppress_small=True))

43680
[[    52.8019       -0.140325     -8.44096       0.019953     12.
  -19339.6         600.          170.            0.          127.669   ]
 [    52.8019       -0.140325     -9.38489       0.019953     12.
  -19347.3         600.          170.            0.          107.073   ]
 [    52.8019       -0.140325     -9.95082       0.019953     12.
  -19267.6         600.          170.            0.           94.524   ]
 [    52.8019       -0.140325    -10.5471        0.019953     12.
  -19190.          600.          170.            0.           81.145   ]
 [    52.8019       -0.140325    -11.1193        0.019953     12.
  -19314.2         600.          170.            0.           68.155   ]
 [    52.8019       -0.140325    -12.0168        0.019953     12.
  -19297.3         600.          170.            0.           47.494   ]
 [    52.8019       -0.140325    -13.0784        0.019953     12.
  -19094.6         600.          170.            0.           22.62    ]
 [    52.8019       -

In [117]:
aux_df = df_train.sort_values(by=['id_simulation', 'srcId','time']).reset_index(drop=True)
aux_df[(aux_df['srcId'] == 0) & (aux_df['id_simulation'] == 'sim_alt_600_raan_170')][['latDev', 'longDev', 'elevSat', 'loraTP', 'loraSF', 'doppler', 'alt', 'raan', 'srcId']].tail(8)
#aux_df.head()

,latDev,longDev,elevSat,loraTP,loraSF,doppler,alt,raan,srcId
52,52.8019,-0.140325,-8.44096,0.019953,12,-19339.6,600,170,0
53,52.8019,-0.140325,-9.38489,0.019953,12,-19347.3,600,170,0
54,52.8019,-0.140325,-9.95082,0.019953,12,-19267.6,600,170,0
55,52.8019,-0.140325,-10.54710,0.019953,12,-19190.0,600,170,0
56,52.8019,-0.140325,-11.11930,0.019953,12,-19314.2,600,170,0
57,52.8019,-0.140325,-12.01680,0.019953,12,-19297.3,600,170,0
58,52.8019,-0.140325,-13.07840,0.019953,12,-19094.6,600,170,0
59,52.8019,-0.140325,-14.02750,0.019953,12,-19077.8,600,170,0


In [64]:
aux_df = df_train.groupby(by=['srcId']).size().reset_index(name='counts')
aux_df

,srcId,counts
0,0,120
1,1,120
2,2,120
3,3,120
4,4,120
5,5,112
6,6,112
7,7,120
8,8,120
9,9,120


In [61]:
aux_df[aux_df['id_simulation'] == 'sim_alt_600_raan_170']

,id_simulation,srcId,counts
0,sim_alt_600_raan_170,0,9
1,sim_alt_600_raan_170,1,9
2,sim_alt_600_raan_170,2,9
3,sim_alt_600_raan_170,3,9
4,sim_alt_600_raan_170,4,9
5,sim_alt_600_raan_170,5,8
6,sim_alt_600_raan_170,6,8
7,sim_alt_600_raan_170,7,9
8,sim_alt_600_raan_170,8,9
9,sim_alt_600_raan_170,9,9
